# Tokenizer BPE para D&D 5e — PHB (inglês) + Manual dos Monstros (português)

Este notebook percorre o caminho completo **PDF → texto limpo → tokens (ids)**. Cada etapa tem uma célula de explicação do conceito, uma célula com o código e, quando faz sentido, uma célula que mostra o efeito nos dados.

| Etapa | O que faz |
|---|---|
| 1 | Extração: PDF → texto bruto |
| 2 | Normalização Unicode |
| 3 | Ruído de página (cabeçalhos, sumário, campos `____`) |
| 4 | Reflow: reconstruir parágrafos |
| 5 | Reparo de OCR (só quando necessário) |
| 6 | Pré-tokenização (regex) |
| 7 | Treino do BPE |
| 8 | Encode / Decode |
| 9 | Avaliação |
| 10 | Exportação (`train.bin` / `val.bin`) |

**Dependências:** só a biblioteca padrão do Python. Para extrair texto dos PDFs é preciso `pdftotext` (pacote `poppler-utils`) **ou** `pip install pypdf`. Para a opção `REOCR = True` também é preciso `tesseract-ocr`.

**Organização esperada:** os dois PDFs dentro da pasta `DATA_DIR` (abaixo). Tudo que o notebook gera vai para `DATA_DIR/processed/`.

## 0. Configuração e utilidades

In [ ]:
from __future__ import annotations

import argparse
import json
import math
import os
import re
import shutil
import subprocess
import sys
import tempfile
import textwrap
import time
import unicodedata
from array import array
from collections import Counter, defaultdict
from dataclasses import dataclass
from multiprocessing.pool import ThreadPool
from pathlib import Path



@dataclass
class Livro:
    arquivo: str  
    idioma: str    
    apelido: str   
    escaneado: bool = False   
    ocr_lang: str = "eng"    


LIVROS = [
    Livro("DD-Player-Handbook.pdf", "en", "PHB", escaneado=True, ocr_lang="eng"),
    Livro("old-dd-5e-manual-dos-monstros-biblioteca-elfica.pdf", "pt", "Monstros"),
]


LETRAS_SOLTAS_VALIDAS = {
    "en": {"a", "i"},
    "pt": {"a", "e", "o", "é", "à", "ó"},
}

FIM_DOC = "<|endoftext|>"         
LIMIAR_OCR_RUIM = 0.05            
FRACAO_VALIDACAO = 20        

_VERBOSE = True   




def info(msg: str) -> None:
    if _VERBOSE:
        print("  → " + msg)


def _curto(s: str, n: int = 100) -> str:
    s = s.replace("\n", "⏎")
    return s if len(s) <= n else s[: n - 1] + "…"

In [ ]:
# ── Ajuste aqui ───────────────────────────────────────────────────────────────
DATA_DIR    = Path("data")           
VOCAB_SIZE  = 4096                  
REOCR       = False                 
REPARAR_OCR = True                   
WORKERS     = os.cpu_count() or 1    
FORCAR_EXTRACAO = False          

OUT_DIR = DATA_DIR / "processed"
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("PDFs esperados em:", DATA_DIR.resolve())
for lv in LIVROS:
    print(f"  {'OK ' if (DATA_DIR / lv.arquivo).exists() else 'FALTA'}  {lv.arquivo}  [{lv.idioma}]")

## Etapa 1 — Extração (PDF → texto)

Um PDF não guarda "texto": guarda **glifos posicionados na página**. Extrair texto é reconstruir a ordem de leitura a partir dessas posições. Aqui há dois casos bem diferentes:

- **PDF digital** (Manual dos Monstros, gerado pelo Word): a camada de texto é fiel.
- **PDF escaneado + OCR** (PHB, gerado pelo ABBYY FineReader): a camada de texto é um *palpite* de um programa de OCR, e vem com erros (`m ultiverse`, `o f`, `com pan ion s`). *Garbage in, garbage out*: um tokenizer treinado em texto sujo gasta vocabulário aprendendo a sujeira.

Para o caso escaneado há duas saídas: **refazer o OCR** a partir da imagem (`REOCR = True`; Tesseract; lento, mas a qualidade é alta) ou **consertar o texto ruim** (Etapa 5; rápido, parcial).

O OCR é **retomável**: cada página pronta é salva em `processed/PHB.ocr_paginas/`. Se você interromper (⏹), rodar a célula de novo continua de onde parou.

In [ ]:
def extrair_texto_pdf(pdf: Path) -> str:
    """Extrai a camada de texto do PDF. Páginas ficam separadas por '\\f' (form feed)."""
    if shutil.which("pdftotext"):  # poppler-utils: mais fiel à ordem de leitura
        r = subprocess.run(["pdftotext", str(pdf), "-"], capture_output=True, check=True)
        return r.stdout.decode("utf-8", errors="replace")
    try:
        from pypdf import PdfReader
    except ImportError:
        sys.exit("Instale o poppler-utils (comando pdftotext) ou rode: pip install pypdf")
    return "\f".join((p.extract_text() or "") for p in PdfReader(str(pdf)).pages)


def _ocr_pagina(args: tuple) -> str:
    """Worker: rasteriza UMA página (pdftoppm) e roda o Tesseract nela. Resultado vai para cache."""
    pdf, pagina, lang, dpi, tmp, cache_dir = args
    arq_cache = Path(cache_dir) / f"p{pagina:04d}.txt"
    if arq_cache.exists():                      
        return arq_cache.read_text(encoding="utf-8")
    base = Path(tmp) / f"p{pagina}"
    subprocess.run(["pdftoppm", "-gray", "-r", str(dpi), "-f", str(pagina), "-l", str(pagina),
                    "-singlefile", pdf, str(base)], check=True)
    img = base.with_suffix(".pgm")
   
    r = subprocess.run(["tesseract", str(img), "stdout", "-l", lang, "--psm", "1"],
                       capture_output=True, env={**os.environ, "OMP_THREAD_LIMIT": "1"})
    img.unlink(missing_ok=True)
    texto = r.stdout.decode("utf-8", errors="replace").replace("\f", "")
    arq_cache.write_text(texto, encoding="utf-8")
    return texto


def ocr_pdf(pdf: Path, lang: str, workers: int, cache_dir: Path, dpi: int = 300) -> str:
    
    for ferramenta in ("pdftoppm", "tesseract", "pdfinfo"):
        if not shutil.which(ferramenta):
            sys.exit(f"--reocr precisa de '{ferramenta}' (apt install poppler-utils tesseract-ocr tesseract-ocr-{lang[:3]})")
    saida = subprocess.run(["pdfinfo", str(pdf)], capture_output=True, text=True, check=True).stdout
    n_pag = int(re.search(r"Pages:\s+(\d+)", saida).group(1))
    cache_dir.mkdir(parents=True, exist_ok=True)
    feitas = len(list(cache_dir.glob("p*.txt")))
    info(f"OCR de {n_pag} páginas com {workers} thread(s) ({feitas} já em cache)")
    t0, textos = time.time(), []
    with tempfile.TemporaryDirectory() as tmp, ThreadPool(workers) as pool:
        tarefas = [(str(pdf), p, lang, dpi, tmp, str(cache_dir)) for p in range(1, n_pag + 1)]
        for i, t in enumerate(pool.imap(_ocr_pagina, tarefas), 1):
            textos.append(t)
            if i % 25 == 0 or i == n_pag:
                dt = time.time() - t0
                info(f"  {i}/{n_pag} páginas — {dt / 60:.1f} min decorridos")
                sys.stdout.flush()
    return "\f".join(textos)


def carregar_texto_bruto(livro: Livro, data_dir: Path, forcar: bool = False,
                         reocr: bool = False, workers: int = 1) -> str:
    """Extrai o texto uma vez e guarda em data/processed/ (cache). Com --reocr, refaz o OCR."""
    usar_ocr = reocr and livro.escaneado
    cache = data_dir / "processed" / f"{livro.apelido}.{'ocr' if usar_ocr else 'bruto'}.txt"
    if cache.exists() and not forcar:
        info(f"[{livro.apelido}] usando cache {cache.name}")
        return cache.read_text(encoding="utf-8")
    pdf = data_dir / livro.arquivo
    if not pdf.exists():
        sys.exit(f"PDF não encontrado: {pdf}")
    t0 = time.time()
    texto = (ocr_pdf(pdf, livro.ocr_lang, workers, data_dir / "processed" / f"{livro.apelido}.ocr_paginas")
             if usar_ocr else extrair_texto_pdf(pdf))
    cache.parent.mkdir(parents=True, exist_ok=True)
    cache.write_text(texto, encoding="utf-8")
    info(f"[{livro.apelido}] {'OCR' if usar_ocr else 'extração'} em {time.time() - t0:.1f}s: "
         f"{texto.count(chr(12)) + 1} páginas, {len(texto):,} caracteres")
    return texto

In [ ]:
brutos = {lv.apelido: carregar_texto_bruto(lv, DATA_DIR, FORCAR_EXTRACAO, REOCR, WORKERS) for lv in LIVROS}

print("\n--- PHB, primeiras linhas do texto bruto (note os espaços falsos) ---")
print(brutos["PHB"][3000:3600])

## Etapa 2 — Normalização Unicode

O mesmo "é" pode ser **1 caractere** (U+00E9) ou **2** (`e` + acento combinante U+0301). Para o BPE, que enxerga **bytes**, são sequências diferentes. A forma **NFC** unifica.

Também trocamos aspas curvas por retas, ligaduras (`ﬁ`) por letras comuns e removemos marcas invisíveis (hífen suave, bullets da fonte Wingdings). Menos variantes = vocabulário mais útil.

> Usamos **NFC** e não NFKC: o NFKC também achataria `½` → `1⁄2` e `²` → `2`, destruindo informação útil num livro de regras.

In [ ]:
LIGADURAS = {"ﬁ": "fi", "ﬂ": "fl", "ﬀ": "ff", "ﬃ": "ffi", "ﬄ": "ffl"}
ASPAS = {"\u2018": "'", "\u2019": "'", "\u201c": '"', "\u201d": '"'}


def normalizar(texto: str) -> str:

    texto = unicodedata.normalize("NFC", texto)

    texto = texto.replace("\u00ad\n", "").replace("\u00ad", "")

    
    for velho, novo in {**LIGADURAS, **ASPAS}.items():
        texto = texto.replace(velho, novo)

    texto = re.sub(r"[\ue000-\uf8ff]", "•", texto)

    
    texto = texto.replace("\u00a0", " ").replace("\u200b", "")
    texto = re.sub(r"[\x00-\x08\x0b\x0e-\x1f\x7f]", "", texto)

    texto = re.sub(r"[^\S\n\f]+", " ", texto)
    return re.sub(r" ?\n ?", "\n", texto)

In [ ]:

a, b = "é", "e\u0301"
print("antes :", a == b, "| bytes:", a.encode(), "vs", b.encode())
print("depois:", normalizar(a) == normalizar(b), "| bytes:", normalizar(b).encode())
print()
print(repr(normalizar("“Don’t” — ﬁre\u00a0ball\uf0b7 item")))

## Etapa 3 — Ruído de página

Cabeçalhos (`52 CAPÍTULO 3 | CLASSES`), números de página, sumário com `..........12` e campos de preencher (`Nome: ________`) **não são linguagem**. E um BPE *adora* padrões repetitivos: sem esta etapa ele gastaria vagas do vocabulário em tokens de `________`.

Detectamos molduras por **frequência**, sem regra específica de livro: uma linha (com os dígitos normalizados) que aparece no topo/rodapé de muitas páginas é moldura, não conteúdo. Exigimos pelo menos um dígito para nunca apagar títulos legítimos que só por acaso caem no topo de uma página (ex.: `AÇÕES`).

**Regra de ouro:** só remova o que você consegue identificar com segurança.

In [ ]:
_RE_LIDER_SUMARIO = re.compile(r"\.{5,}")   
_RE_CAMPO_PREENCHER = re.compile(r"_{3,}")   

def _forma(linha: str) -> str:
    return re.sub(r"\d+", "#", linha)


def limpar_paginas(texto: str) -> tuple[str, Counter]:
    
    removidos: Counter = Counter()
    paginas = [[l.strip() for l in p.split("\n")] for p in texto.split("\f")]

    bordas: Counter = Counter()
    for linhas in paginas:
        nao_vazias = [l for l in linhas if l]
        for l in {nao_vazias[0], nao_vazias[-1]} if nao_vazias else ():
            if len(l) <= 60 and re.search(r"\d", l):
                bordas[_forma(l)] += 1
    limite = max(5, int(0.03 * len(paginas)))
    molduras = {f for f, n in bordas.items() if n >= limite}

    saida = []
    for linhas in paginas:
        nao_vazias = [i for i, l in enumerate(linhas) if l]
        for i in {nao_vazias[0], nao_vazias[-1]} if nao_vazias else ():
            if _forma(linhas[i]) in molduras:
                removidos["cabeçalhos/rodapés/nº de página"] += 1
                linhas[i] = ""
        mantidas = []
        for l in linhas:
            if _RE_LIDER_SUMARIO.search(l):
                removidos["linhas de sumário"] += 1
                continue
            if _RE_CAMPO_PREENCHER.search(l):
                removidos["campos '____' colapsados"] += 1
                l = _RE_CAMPO_PREENCHER.sub("___", l)
            mantidas.append(l)
        saida.append("\n".join(mantidas))
    return "\n\n".join(saida), removidos

In [ ]:
for l in ["52 CAPÍTULO 3 | CLASSES", "53 CAPÍTULO 3 | CLASSES", "Pontos de Vida 39"]:
    print(f"{l!r:34} → {_forma(l)!r}")

## Etapa 4 — Reflow (reconstruir parágrafos)

O PDF quebra linhas por **largura de página**, não por sentido. Se mantivermos essas quebras, o modelo aprende a "pular linha a cada ~55 caracteres": *layout*, não linguagem. Colamos as linhas que são continuação de frase e mantemos as outras.

Stat blocks (`Pontos de Vida 39 (6d10 + 6)`) ficam intactos: cada linha começa com maiúscula, então nunca são coladas.

In [ ]:
def reflow(texto: str) -> str:
    
    paragrafos: list[str] = []
    atual, houve_branco = "", False
    for linha in texto.split("\n"):
        linha = linha.strip()
        if not linha:
            houve_branco = True
            continue
        if not atual:
            atual, houve_branco = linha, False
            continue
        if houve_branco:
            junta = linha[0].islower() and atual[-1] not in '.!?:;"”)'
        else:
            junta = linha[0].islower() or atual[-1] in ",;-—("
        if junta:
            atual += " " + linha
        else:
            paragrafos.append(atual)
            atual = linha
        houve_branco = False
    if atual:
        paragrafos.append(atual)
    return "\n".join(paragrafos)

In [ ]:
demo = """O ankheg usa suas poderosas mandíbulas para cavar
sinuosos túneis nas profundezas do solo.
Classe de Armadura 14 (armadura natural)
Pontos de Vida 39 (6d10 + 6)"""
print(reflow(demo))

## Etapa 5 — Reparo de OCR (só quando necessário)

Medimos a **fragmentação**: % de palavras que são uma letra solta inválida (`m`, `w`, `f`). Se passar do limiar (5%), reparamos com um **modelo de linguagem de unigramas construído do próprio texto** + **programação dinâmica (Viterbi)**, que escolhe para cada parágrafo a segmentação de menor custo `-log P(palavra)`. Palavra inteira conhecida é barata; fragmento desconhecido é caro.

Duas salvaguardas contra falsas fusões:

1. peças de continuação precisam começar em minúscula (impede `N` + `NG` → `NNG`);
2. colar duas palavras *válidas* (`in` + `to`) exige vantagem estatística grande.

**Limite fundamental:** palavras que **nunca** aparecem inteiras no texto (`temples`, `remain`) não podem ser recuperadas, pois o vocabulário vem do próprio texto. Para isso o certo é refazer o OCR (`REOCR = True`). Esta etapa é o plano B, rápido.

In [ ]:
_TOKEN = re.compile(r"^([^\w'-]*)('?[^\W\d_]+(?:['-][^\W\d_]+)*)([^\w'-]*)$")


def _analisar(tok: str):
    m = _TOKEN.match(tok)
    return m.groups() if m else None


def taxa_fragmentacao(texto: str, idioma: str) -> float:
    validas = LETRAS_SOLTAS_VALIDAS[idioma]
    palavras = re.findall(r"[^\W\d_]+", texto)
    if not palavras:
        return 0.0
    return sum(1 for w in palavras if len(w) == 1 and w.lower() not in validas) / len(palavras)


def reparar_ocr(texto: str, idioma: str, passes: int = 2, janela: int = 7,
                min_count: int = 2, margem: float = 3.0) -> tuple[str, list]:
    validas_soltas = LETRAS_SOLTAS_VALIDAS[idioma]
    exemplos: list[tuple[str, str]] = []
    custo_oov_base, custo_oov_char = 12.0, 2.0   # "palavra" desconhecida: cara e cresce com o tamanho

    for passo in range(passes):
        contagem: Counter = Counter()
        for tok in texto.split():
            p = _analisar(tok)
            if p:
                contagem[p[1].lower()] += 1
        vocab = {w: n for w, n in contagem.items() if n >= min_count}
        total = sum(contagem.values())

        def valida(w: str) -> bool:
            return w in vocab and (len(w) > 1 or w in validas_soltas)

        def custo(w: str) -> float:
            if valida(w):
                return -math.log(vocab[w] / total)
            return custo_oov_base + custo_oov_char * len(w)

        linhas_saida = []
        for linha in texto.split("\n"):
            toks = linha.split()
            n = len(toks)
            if n < 2:
                linhas_saida.append(linha)
                continue
            P = [_analisar(t) for t in toks]
            melhor = [0.0] + [math.inf] * n
            volta = [0] * (n + 1)
            for j in range(1, n + 1):
                for i in range(max(0, j - janela), j):
                    seg = P[i:j]
                    if j - i == 1:
                        c = custo(seg[0][1].lower()) if seg[0] else 0.0
                    else:
                        if any(s is None for s in seg):
                            continue
                        # nada de colar através de pontuação ("fim. Começo")
                        if any(s[2] for s in seg[:-1]) or any(s[0] for s in seg[1:]):
                            continue
                        # peças de continuação começam em minúscula (ou apóstrofo/hífen).
                        # Isso impede "N" + "NG" -> "NNG" e "A" + "Bola" -> "ABola".
                        if not all(s[1][0].islower() or s[1][0] in "'-" for s in seg[1:]):
                            continue
                        pecas = [s[1].lower() for s in seg]
                        colado = "".join(pecas)
                        custo_pecas = sum(custo(x) for x in pecas)
                        algum_ruim = any(not valida(x) for x in pecas)
                        if colado in vocab and len(colado) > 1:
                            c = custo(colado) if valida(colado) else custo_oov_base
                            # Se todas as peças já são palavras válidas ("in" + "to"), só colamos
                            # com ampla vantagem estatística — evita "into" onde era "in to".
                            if not algum_ruim and not (c + margem < custo_pecas):
                                continue
                        elif algum_ruim and all(not valida(x) for x in pecas) \
                                and any(len(x) == 1 for x in pecas) and len(colado) >= 3:
                            # Palavra rara nunca vista inteira ("m ortal"): só cola se TODAS
                            # as peças forem lixo e houver uma letra solta no meio.
                            c = custo_oov_base + custo_oov_char * len(colado)
                        else:
                            continue
                        if c >= custo_pecas:
                            continue
                    if melhor[i] + c < melhor[j]:
                        melhor[j], volta[j] = melhor[i] + c, i
            partes, j = [], n
            while j > 0:
                i = volta[j]
                partes.append("".join(toks[i:j]))
                j = i
            nova = " ".join(reversed(partes))
            if passo == 0 and nova != " ".join(toks) and len(exemplos) < 4 and 60 < len(linha) < 400:
                exemplos.append((linha, nova))
            linhas_saida.append(nova)
        texto = "\n".join(linhas_saida)
    return texto, exemplos

### Executando as etapas 2 a 5 nos dois livros

`preparar_livro` encadeia normalização → ruído de página → reflow → (reparo de OCR, se a fragmentação passar do limiar) e imprime o que cada etapa mudou.

In [ ]:
def preparar_livro(livro: Livro, bruto: str, reparar: bool, usou_reocr: bool = False) -> str:
    
    n0 = len(bruto)

    texto = normalizar(bruto)
    info(f"[{livro.apelido}] ETAPA 2 normalização:  {n0:,} → {len(texto):,} caracteres")

    texto, removidos = limpar_paginas(texto)
    resumo = ", ".join(f"{v} {k}" for k, v in removidos.items()) or "nada"
    info(f"[{livro.apelido}] ETAPA 3 ruído de página: removido {resumo}")

    linhas_antes = texto.count("\n")
    texto = reflow(texto)
    info(f"[{livro.apelido}] ETAPA 4 reflow:          {linhas_antes:,} linhas → {texto.count(chr(10)) + 1:,} parágrafos")

    taxa = taxa_fragmentacao(texto, livro.idioma)
    if reparar and taxa > LIMIAR_OCR_RUIM:
        info(f"[{livro.apelido}] ETAPA 5 reparo de OCR:   fragmentação {taxa:.1%} > {LIMIAR_OCR_RUIM:.0%} → reparando…")
        t0 = time.time()
        texto, exemplos = reparar_ocr(texto, livro.idioma)
        info(f"[{livro.apelido}]   fragmentação depois: {taxa_fragmentacao(texto, livro.idioma):.1%} "
             f"({time.time() - t0:.0f}s)")
        for antes, depois in exemplos[:3]:
            info("  antes:  " + _curto(antes))
            info("  depois: " + _curto(depois))
        if livro.escaneado and not usou_reocr:
            info(f"[{livro.apelido}] DICA: o reparo é parcial. Para qualidade muito melhor rode com --reocr "
                 f"(precisa de tesseract-ocr; uma vez só, fica em cache).")
    else:
        info(f"[{livro.apelido}] ETAPA 5 reparo de OCR:   fragmentação {taxa:.1%} — nada a reparar")
    return texto

In [ ]:
limpos = {lv.apelido: preparar_livro(lv, brutos[lv.apelido], REPARAR_OCR, REOCR) for lv in LIVROS}

for apelido, t in limpos.items():
    (OUT_DIR / f"{apelido}.limpo.txt").write_text(t, encoding="utf-8")
print(f"\ntextos limpos salvos em {OUT_DIR}/*.limpo.txt  (abra e confira com os seus olhos!)")

In [ ]:
import random
random.seed(7)
for apelido, t in limpos.items():
    longos = [p for p in t.split("\n") if 150 < len(p) < 400]
    print(f"\n=== {apelido}: 3 parágrafos aleatórios do texto limpo ===")
    for p in random.sample(longos, 3):
        print("•", p, "\n")

## Etapa 6 — Pré-tokenização

Antes do BPE, uma **regex** corta o texto em "pedaços", e o BPE **nunca funde através da fronteira de um pedaço**. Efeito: não existem tokens que misturam o fim de uma palavra com o início da outra, e a pontuação não gruda nas palavras.

Decisões deste projeto:

- o **espaço fica no início** da palavra (`" dragão"`), então `dragão` e ` dragão` são tokens diferentes: o modelo aprende onde as palavras começam;
- **dígitos entram um por vez** (`243` → `2`,`4`,`3`): stat blocks como `18d10 + 144` ficam regulares;
- letras usam a classe Unicode, então `ã`, `ç`, `é` contam como letras.

In [ ]:
PRE_TOKENIZACAO = re.compile(
    r"'(?:[sdmt]|ll|ve|re)"    
    r"| ?[^\W\d_]+"            
    r"| ?\d"                    
    r"| ?(?:[^\s\w]|_)+"       
    r"|\s+(?!\S)"               
    r"|\s+"          
)

In [ ]:
exemplo = "Pontos de Vida 243 (18d10 + 144) — can't"
print(" | ".join(m.group().replace(" ", "·") for m in PRE_TOKENIZACAO.finditer(exemplo)))

## Etapa 7 — Treino do BPE (byte-level)

Começamos com **256 tokens** (um por byte) e repetimos: achar o **par de vizinhos mais frequente** e fundi-lo num token novo. Os primeiros merges são pares comuns (`" t"`, `"es"`); os últimos viram palavras e sílabas inteiras.

- **Byte-level + UTF-8:** `ã` ocupa 2 bytes, então começa como 2 tokens e só vira 1 se for frequente o bastante para merecer uma fusão. Como todo texto é representável em bytes, **nunca existe `[UNK]`**.
- **Vocabulário compartilhado por dois idiomas:** as fusões se dividem entre inglês (PHB) e português (Monstros), proporcionalmente ao que há em cada corpus.
- **Trade-off central:** vocabulário maior → sequências mais curtas (o modelo vê mais contexto), mas matriz de embeddings maior e tokens raros mal treinados.
- **Eficiência:** o BPE trabalha sobre pré-tokens *únicos* com peso (`" the"` aparece 20 mil vezes, mas é processado uma vez) e atualiza só as palavras afetadas por cada fusão.

Separamos **1 a cada 20 parágrafos** para validação. O BPE nunca os vê no treino, e isso permite medir se o vocabulário generaliza.

In [ ]:
class BPETokenizer:
    
    def __init__(self) -> None:
        self.merges: list[tuple[int, int]] = []
        self.vocab: dict[int, bytes] = {i: bytes([i]) for i in range(256)}
        self.especiais: dict[str, int] = {}
        self._rank: dict[tuple[int, int], int] = {}
        self._cache: dict[bytes, list[int]] = {}

    @property
    def tamanho_vocab(self) -> int:
        return len(self.vocab) + len(self.especiais)

    def treinar(self, textos: list[str], vocab_size: int, verbose: bool = True) -> None:
        n_fusoes = vocab_size - 256 - 1   # -1 reservado para <|endoftext|>
        if n_fusoes <= 0:
            raise ValueError("vocab_size precisa ser > 257")

        
        freq: Counter = Counter()
        for t in textos:
            for m in PRE_TOKENIZACAO.finditer(t):
                freq[m.group().encode("utf-8")] += 1
        palavras = [list(b) for b in freq]          
        pesos = list(freq.values())
        if verbose:
            info(f"{sum(pesos):,} pré-tokens no total, {len(palavras):,} distintos")

        pares: Counter = Counter()
        onde: dict[tuple[int, int], set[int]] = defaultdict(set)
        for idx, seq in enumerate(palavras):
            for p in zip(seq, seq[1:]):
                pares[p] += pesos[idx]
                onde[p].add(idx)

        t0 = time.time()
        for k in range(n_fusoes):
            if not pares:
                break
            par = max(pares, key=lambda p: (pares[p], p))  
            if pares[par] < 2:
                break                                       
            novo_id = 256 + k
            self.merges.append(par)
            self.vocab[novo_id] = self.vocab[par[0]] + self.vocab[par[1]]
            if verbose and (k < 12 or (k + 1) % 500 == 0):
                tok = self.vocab[novo_id].decode("utf-8", errors="replace")
                info(f"fusão {k + 1:>5}: {self.vocab[par[0]]!r:>12} + {self.vocab[par[1]]!r:<12} "
                     f"→ {tok!r:<14} ({pares[par]:,}x)")
            for idx in list(onde[par]):
                seq, w = palavras[idx], pesos[idx]
                for p in zip(seq, seq[1:]):                 
                    pares[p] -= w
                    if pares[p] <= 0:
                        del pares[p]
                nova, i = [], 0
                while i < len(seq):
                    if i < len(seq) - 1 and seq[i] == par[0] and seq[i + 1] == par[1]:
                        nova.append(novo_id)
                        i += 2
                    else:
                        nova.append(seq[i])
                        i += 1
                palavras[idx] = nova
                for p in zip(nova, nova[1:]):            
                    pares[p] += w
                    onde[p].add(idx)
            pares.pop(par, None)
            onde.pop(par, None)

        self.especiais = {FIM_DOC: len(self.vocab)}
        self._rank = {p: i for i, p in enumerate(self.merges)}
        self._cache.clear()
        if verbose:
            info(f"treino concluído em {time.time() - t0:.1f}s — vocabulário final: {self.tamanho_vocab}")

    # ── ETAPA 8: encode / decode ──────────────────────────────────────────────
    def _bpe(self, b: bytes) -> list[int]:
        """Aplica as fusões, da MAIS ANTIGA (mais frequente no treino) para a mais nova."""
        if b in self._cache:
            return self._cache[b]
        ids = list(b)
        while len(ids) >= 2:
            candidatos = set(zip(ids, ids[1:]))
            par = min(candidatos, key=lambda p: self._rank.get(p, math.inf))
            if par not in self._rank:
                break                                     
            novo_id, nova, i = 256 + self._rank[par], [], 0
            while i < len(ids):
                if i < len(ids) - 1 and (ids[i], ids[i + 1]) == par:
                    nova.append(novo_id)
                    i += 2
                else:
                    nova.append(ids[i])
                    i += 1
            ids = nova
        self._cache[b] = ids
        return ids

    def encode(self, texto: str, permitir_especiais: bool = True) -> list[int]:
        ids: list[int] = []
        if permitir_especiais and self.especiais:
            padrao = "(" + "|".join(re.escape(s) for s in self.especiais) + ")"
            partes = re.split(padrao, texto)
        else:
            partes = [texto]
        for parte in partes:
            if parte in self.especiais and permitir_especiais:
                ids.append(self.especiais[parte])
                continue
            for m in PRE_TOKENIZACAO.finditer(parte):
                ids.extend(self._bpe(m.group().encode("utf-8")))
        return ids

    def decode(self, ids: list[int]) -> str:
        inv_esp = {v: k.encode("utf-8") for k, v in self.especiais.items()}
        dados = b"".join(inv_esp[i] if i in inv_esp else self.vocab[i] for i in ids)
        return dados.decode("utf-8", errors="replace")

    def visualizar(self, texto: str) -> str:
        """Mostra os pedaços: '·' = espaço, '⏎' = quebra de linha, '�' = byte parcial de caractere UTF-8."""
        partes = [self.vocab[i].decode("utf-8", errors="replace") for i in self.encode(texto)]
        return " | ".join(p.replace(" ", "·").replace("\n", "⏎") for p in partes)

    def salvar(self, caminho: Path) -> None:
        caminho.parent.mkdir(parents=True, exist_ok=True)
        caminho.write_text(json.dumps({
            "tipo": "bpe-byte-level",
            "pre_tokenizacao": PRE_TOKENIZACAO.pattern,
            "especiais": self.especiais,
            "merges": [list(m) for m in self.merges],
        }, ensure_ascii=False), encoding="utf-8")

    @classmethod
    def load(cls, caminho: str | Path) -> "BPETokenizer":
        dados = json.loads(Path(caminho).read_text(encoding="utf-8"))
        tok = cls()
        tok.merges = [tuple(m) for m in dados["merges"]]
        for i, (a, b) in enumerate(tok.merges):
            tok.vocab[256 + i] = tok.vocab[a] + tok.vocab[b]
        tok.especiais = dados["especiais"]
        tok._rank = {p: i for i, p in enumerate(tok.merges)}
        return tok

In [ ]:
def dividir_treino_validacao(texto: str) -> tuple[str, str]:
    paragrafos = texto.split("\n")
    treino = [p for i, p in enumerate(paragrafos) if i % FRACAO_VALIDACAO]
    valid = [p for i, p in enumerate(paragrafos) if i % FRACAO_VALIDACAO == 0]
    return "\n".join(treino), "\n".join(valid)

split = {lv.apelido: dividir_treino_validacao(limpos[lv.apelido]) for lv in LIVROS}

tok = BPETokenizer()
tok.treinar([split[lv.apelido][0] for lv in LIVROS], VOCAB_SIZE, verbose=True)
tok.salvar(OUT_DIR / "tokenizer.json")

## Etapa 8 — Encode / Decode

- **Encode:** pré-tokeniza, converte cada pedaço em bytes e aplica as fusões na **ordem em que foram aprendidas** (a mais frequente primeiro).
- **Decode:** concatena os bytes de cada id e decodifica UTF-8.

Como o vocabulário cobre todos os 256 bytes, o round-trip `decode(encode(x)) == x` é **sempre exato**. Abaixo, `·` = espaço e `�` = byte parcial de um caractere multibyte (acentos podem ser "partidos" em dois tokens).

In [ ]:
amostras = [
    "The wizard casts fireball at the dragon.",
    "O mago conjura bola de fogo no dragão.",
    "Pontos de Vida 243 (18d10 + 144)",
    "Melee Weapon Attack: +15 to hit, reach 5 ft.",
]
for s in amostras:
    ids = tok.encode(s)
    assert tok.decode(ids) == s          # round-trip exato
    print(f"{s}\n    {len(ids)} tokens: {tok.visualizar(s)}\n")

## Etapa 9 — Avaliação

Métricas que importam:

- **bytes/token:** quanto maior, melhor a compressão (1.0 = nenhuma; só bytes crus).
- **tokens/palavra** ("fertilidade"): ~1.0 é ótimo; >2 indica que o tokenizer fragmenta demais aquele idioma. Costuma ser pior no idioma menos representado. Aqui já vem inflado porque cada **dígito** é um token (stat blocks têm muitos números).
- **Validação:** medimos em parágrafos que o BPE **nunca viu** no treino. Se a compressão cair muito do treino para a validação, o vocabulário decorou.
- **round-trip:** `decode(encode(x)) == x` tem que dar `OK`.

In [ ]:
def avaliar(tok: BPETokenizer, textos: dict[str, str]) -> None:
    info(f"{'conjunto':<18}{'tokens':>10}{'bytes/token':>13}{'chars/token':>13}{'tokens/palavra':>16}  round-trip")
    for nome, t in textos.items():
        ids = tok.encode(t)
        n_bytes, n_pal = len(t.encode("utf-8")), max(1, len(re.findall(r"\w+", t)))
        ok = "OK" if tok.decode(ids) == t else "FALHOU"
        info(f"{nome:<18}{len(ids):>10,}{n_bytes / len(ids):>13.2f}{len(t) / len(ids):>13.2f}"
             f"{len(ids) / n_pal:>16.2f}  {ok}")

In [ ]:
conjuntos = {}
for lv in LIVROS:
    conjuntos[f"{lv.apelido} (treino)"] = split[lv.apelido][0]
    conjuntos[f"{lv.apelido} (valid.)"] = split[lv.apelido][1]
avaliar(tok, conjuntos)

usados = Counter()
for lv in LIVROS:
    usados.update(tok.encode(split[lv.apelido][1]))
nao_vistos = sum(1 for i in range(tok.tamanho_vocab - 1) if i not in usados)
info(f"tokens nunca usados na validação: {nao_vistos} de {tok.tamanho_vocab} (raros/específicos de um só livro)")
longos = sorted((v.decode("utf-8", "replace") for k, v in tok.vocab.items() if k >= 256), key=len, reverse=True)[:12]
info("maiores tokens aprendidos: " + ", ".join(repr(x) for x in longos))

## Etapa 10 — Exportação para o modelo

O modelo não lê texto: lê **arrays de inteiros**. Convertemos cada livro em ids, colocamos `<|endoftext|>` entre os livros (para o modelo saber que o contexto acabou) e gravamos como **uint16** (2 bytes/token, suficiente para vocabulários < 65.536). Treino e validação ficam em arquivos separados.

In [ ]:
assert tok.tamanho_vocab < 65536
for nome, indice in (("train", 0), ("val", 1)):
    ids: list[int] = []
    for lv in LIVROS:
        ids.extend(tok.encode(split[lv.apelido][indice]))
        ids.append(tok.especiais[FIM_DOC])
    with open(OUT_DIR / f"{nome}.bin", "wb") as f:
        array("H", ids).tofile(f)
    info(f"{nome}.bin: {len(ids):,} tokens")
info(f"tokenizer salvo em {OUT_DIR / 'tokenizer.json'}")

## Usando o tokenizer depois de treinado

O `tokenizer.json` guarda só a lista de fusões. Carregar é rápido, e não precisa refazer nada das etapas anteriores. Os `.bin` se leem de volta com `array("H")` (ou `numpy.fromfile(..., dtype=np.uint16)`).

In [ ]:
tok2 = BPETokenizer.load(OUT_DIR / "tokenizer.json")
ids = tok2.encode("O dragão vermelho ataca! The red dragon attacks!")
print(ids)
print(tok2.decode(ids))

dados = array("H")
with open(OUT_DIR / "val.bin", "rb") as f:
    dados.frombytes(f.read())
print(f"\nval.bin: {len(dados):,} tokens. Primeiros 40 decodificados:\n", tok2.decode(list(dados[:40])))